# **Project Title: Bert for AG News**

# Submitted by: Afeera Sagheer

# **1_ Importing Libraries**

In [1]:
# auto tokenizer to select best best tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print("Tokenizer loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully


In [2]:
# AutoModelForSequenceClassification for text classification
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=4)
print("Model loaded successfully")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully


# **2_ Loading Dataset**

In [17]:
# loading dataset and making dataframe
import pandas as pd
url = "https://raw.githubusercontent.com/mhjabreel/CharCNN_Keras/master/data/ag_news_csv/train.csv"
df = pd.read_csv(url,header=None,names=["label", "title", "description"])
print(df.head())

   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   
1      3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2      3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
3      3  Iraq Halts Oil Exports from Main Southern Pipe...   
4      3  Oil prices soar to all-time record, posing new...   

                                         description  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  
3  Reuters - Authorities have halted oil export\f...  
4  AFP - Tearaway world oil prices, toppling reco...  


In [4]:
# seperating training nad testing df
import pandas as pd
train_url = "https://raw.githubusercontent.com/mhjabreel/CharCNN_Keras/master/data/ag_news_csv/train.csv"
test_url = "https://raw.githubusercontent.com/mhjabreel/CharCNN_Keras/master/data/ag_news_csv/test.csv"
train_df = pd.read_csv(train_url,header=None,names=["label", "title", "description"])
test_df = pd.read_csv(test_url,header=None,names=["label", "title", "description"])
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"] = test_df["title"] + " " + test_df["description"]

print(train_df.shape)
print(test_df.shape)

(120000, 4)
(7600, 4)


# **3_ Data Inspection**

In [18]:
# checking first five rows
df.head()

,label,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [19]:
# checking last five rows
df.tail()

,label,title,description
119995,1,Pakistan's Musharraf Says Won't Quit as Army C...,KARACHI (Reuters) - Pakistani President Perve...
119996,2,Renteria signing a top-shelf deal,Red Sox general manager Theo Epstein acknowled...
119997,2,Saban not going to Dolphins yet,The Miami Dolphins will put their courtship of...
119998,2,Today's NFL games,PITTSBURGH at NY GIANTS Time: 1:30 p.m. Line: ...
119999,2,Nets get Carter from Raptors,INDIANAPOLIS -- All-Star Vince Carter was trad...


In [20]:
# checking percentile, min , max ,std
df.describe()

,label
count,120000.000000
mean,2.500000
std,1.118039
min,1.000000
25%,1.750000
50%,2.500000
75%,3.250000
max,4.000000


In [21]:
# checking info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   label        120000 non-null  int64 
 1   title        120000 non-null  object
 2   description  120000 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.7+ MB


In [22]:
# checking number of rows and columns
df.shape

(120000, 3)

In [23]:
# checking number of enteries
df.size

360000

In [24]:
# checking column names
df.columns

Index(['label', 'title', 'description'], dtype='object')

In [25]:
# checking datatypes
df.dtypes

,0
label,int64
title,object
description,object


# **4_ Data Cleaning**

In [26]:
# checking null enteries in each column
df.isnull()

,label,title,description
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
119995,False,False,False
119996,False,False,False
119997,False,False,False
119998,False,False,False


In [27]:
# checking null enteries in whole dataset
df.isnull().sum()

,0
label,0
title,0
description,0


In [28]:
# checking not null enteries in each column
df.notnull()

,label,title,description
0,True,True,True
1,True,True,True
2,True,True,True
3,True,True,True
4,True,True,True
...,...,...,...
119995,True,True,True
119996,True,True,True
119997,True,True,True
119998,True,True,True


In [29]:
# checking not null enteries in whole dataset
df.notnull().sum()

,0
label,120000
title,120000
description,120000


In [30]:
# it drops missing columns
df.dropna()

,label,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."
...,...,...,...
119995,1,Pakistan's Musharraf Says Won't Quit as Army C...,KARACHI (Reuters) - Pakistani President Perve...
119996,2,Renteria signing a top-shelf deal,Red Sox general manager Theo Epstein acknowled...
119997,2,Saban not going to Dolphins yet,The Miami Dolphins will put their courtship of...
119998,2,Today's NFL games,PITTSBURGH at NY GIANTS Time: 1:30 p.m. Line: ...


In [31]:
# it fills missing values
df.interpolate()

/tmp/ipykernel_3830/603973446.py:2: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df.interpolate()


,label,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."
...,...,...,...
119995,1,Pakistan's Musharraf Says Won't Quit as Army C...,KARACHI (Reuters) - Pakistani President Perve...
119996,2,Renteria signing a top-shelf deal,Red Sox general manager Theo Epstein acknowled...
119997,2,Saban not going to Dolphins yet,The Miami Dolphins will put their courtship of...
119998,2,Today's NFL games,PITTSBURGH at NY GIANTS Time: 1:30 p.m. Line: ...


# **5_ Numerical Operations**

In [32]:
# checking average in each column
df.mean(numeric_only = True)

,0
label,2.5


In [33]:
# checking average in whole dataset
df.mean(numeric_only= True).sum()

np.float64(2.5)

In [34]:
# checking middle value in each column
df.median(numeric_only = True)

,0
label,2.5


In [35]:
# checking middle value in whole dataset
df.median(numeric_only= True).sum()

np.float64(2.5)

In [36]:
# checking most repeated value in each column
df.mode(numeric_only = True)

,label
0,1
1,2
2,3
3,4


In [37]:
# checking most repeated value in whole dataset
df.mode(numeric_only= True).sum()

,0
label,10


In [38]:
# checking spread of values from mean in each column
df.std(numeric_only = True)

,0
label,1.118039


In [39]:
# checking spread of values from mean in whole dataset
df.std(numeric_only= True).sum()

np.float64(1.1180386472539636)

In [40]:
# varriance is square of std
df.var(numeric_only = True)

,0
label,1.25001


In [41]:
# checking maximum value
df.max(numeric_only = True)

,0
label,4


In [42]:
# Checking minimum value
df.min(numeric_only = True)

,0
label,1


# **6_ Train Test Split**

In [43]:
# seperating training and testing data
train_df = train_df.sample(n=20000,random_state=42)
test_df = test_df.sample(n=3000,random_state=42)

In [6]:
train_df["label"] = train_df["label"] - 1
test_df["label"] = test_df["label"] - 1

# **7_Tokenizer**

In [44]:
# auto tokenizer selects the best tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [47]:
import torch
# importing news dataset
from torch.utils.data import Dataset
class NewsDataset(Dataset):
    # constructing the dataset
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    # dataset length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        # Tokenization
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt")
        # creating output dictionary
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx],dtype=torch.long)}

        return item

# **8_ Model Training**

In [48]:
# creating list of news
train_dataset = NewsDataset(train_df["text"].tolist(),train_df["label"].tolist())
# creating list of class
test_dataset = NewsDataset(test_df["text"].tolist(),test_df["label"].tolist())

In [10]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=4)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# giving the command of hoe to train model
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none")

# **9_Evaluation**

In [49]:
# checking accuracy
from sklearn.metrics import accuracy_score
import numpy as np
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions,axis=1)
    acc = accuracy_score(labels,preds)
    return {"accuracy": acc}

In [50]:
# training model
from transformers import Trainer
trainer = Trainer(model=model,args=training_args,train_dataset=train_dataset,eval_dataset=test_dataset,compute_metrics=compute_metrics)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.341298,0.268760,0.919000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2500, training_loss=0.3412978271484375, metrics={'train_runtime': 275.4088, 'train_samples_per_second': 72.619, 'train_steps_per_second': 9.077, 'total_flos': 662360616960000.0, 'train_loss': 0.3412978271484375, 'epoch': 1.0})

In [15]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
0.341298,0.268760,1,0.919000


{'eval_loss': 0.268759548664093, 'eval_accuracy': 0.919}

In [51]:
import torch
text = "Apple launches a new AI-powered smartphone."
inputs = tokenizer(text,return_tensors="pt",truncation=True,padding=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Move inputs to same device as model
inputs = {k: v.to(device) for k, v in inputs.items()}
model.to(device)
with torch.no_grad():
    outputs = model(**inputs)
prediction = outputs.logits.argmax(dim=1).item()
labels = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]
print("Prediction:", labels[prediction])

Prediction: Sci/Tech
